# DS23 · Session 44 · Multi-Agent Systems 1/3
## Protocols, Orchestrator-Worker, and LangGraph

**BIU DS23 by Shlomit Levavi**

We split our single agent into a team. First we build a two-agent team (researcher + writer)
**by hand**, with a structured handoff and a validated schema. Then we rebuild the exact same pair
in **LangGraph**, and see that the manual loop was a graph all along.

**The punchline stands:** *Autonomy is a budget, not a switch.* A team is more autonomy, so it must
earn its cost. We measure the topology cost, and honestly predict the S46 measurement.


### Setup: backend, pricing, and the budget guard
Same spine as S42-S43. The guard is locked at the top of every agent notebook.

In [1]:
import os, json, re, math
from datetime import date, timedelta

SEED = 42
BACKEND = "groq"
MODEL   = "llama-3.3-70b-versatile"

MAX_ITERATIONS = 8
COST_CAP_USD   = 0.10
PRICE_IN  = 0.59 / 1_000_000
PRICE_OUT = 0.79 / 1_000_000

USE_LIVE = False   # flip to True in class after verifying the Groq key in Colab Secrets

def estimate_tokens(text):
    return max(1, math.ceil(len(str(text)) / 4))

print("Backend:", BACKEND, "| guard:", MAX_ITERATIONS, "iters,", COST_CAP_USD, "USD")

Backend: groq | guard: 8 iters, 0.1 USD


## Part A · Two agents, by hand

### Tools and a deterministic chat stub
The researcher gets `search_regulation` (read-only). The writer gets no tools. Offline we use a
deterministic stub so the notebook is reproducible; in class `USE_LIVE=True` calls Groq.

In [2]:
# researcher's only tool: read-only retrieval (least privilege by role)
_REG = {"gdpr breach": ("Article 33: notify within 72 hours of awareness.", [33, 34]),
        "ai act high-risk": ("Article 16: providers of high-risk AI ensure conformity.", [16])}

def search_regulation(query: str, regulation: str) -> str:
    """Read-only retrieval from GDPR / AI Act. The researcher's only tool."""
    for key, (text, ids) in _REG.items():
        if any(w in query.lower() for w in key.split()):
            return json.dumps({"passage": text, "article_ids": ids})
    return json.dumps({"passage": "no match", "article_ids": []})

# a deterministic per-role stub: returns what each persona would produce
def role_llm(role, prompt, state):
    """Offline deterministic controller standing in for a per-persona LLM call."""
    if USE_LIVE:
        from groq import Groq
        from google.colab import userdata
        client = Groq(api_key=userdata.get("GROQ_API_KEY"))
        msgs = [{"role": "system", "content": prompt}, {"role": "user", "content": json.dumps(state)}]
        r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0)
        return r.choices[0].message.content
    # offline behaviour per role
    if role == "researcher":
        hit = json.loads(search_regulation("gdpr breach notification", "GDPR"))
        return json.dumps({"findings": [{"claim": hit["passage"], "article_ids": hit["article_ids"]}],
                           "confidence": "high"})
    if role == "writer":
        findings = state.get("findings", [])
        claim = findings[0]["claim"] if findings else "(no findings)"
        ids = findings[0]["article_ids"] if findings else []
        return f"Under GDPR {claim} (see articles {ids})."
    if role == "judge":
        draft = state.get("draft", "")
        ok = "Article 33" in draft or "articles [33" in draft
        return json.dumps({"grounded": ok})
    return ""

print("tools + role stub ready. researcher is read-only.")

tools + role stub ready. researcher is read-only.


### The handoff schema (a contract between agents)
The message from researcher to writer is structured, and validated before it passes, exactly like
tool arguments in S42. A broken handoff is caught early, not in the final report.

In [3]:
def make_handoff(sender, recipient, findings, confidence):
    """Build a structured handoff message between two agents."""
    return {"from": sender, "to": recipient, "findings": findings, "confidence": confidence}

def assert_handoff(handoff, tool_result_ids):
    """Assert v3, at the team level: schema + grounding of the handoff."""
    # 1. schema: required fields present
    for field in ("from", "to", "findings", "confidence"):
        assert field in handoff, f"handoff missing field: {field}"
    assert isinstance(handoff["findings"], list) and handoff["findings"], "findings must be non-empty"
    # 2. grounding: every cited article id actually came from the tool
    for f in handoff["findings"]:
        for aid in f.get("article_ids", []):
            assert aid in tool_result_ids, f"ungrounded article id in handoff: {aid}"
    return True

# quick check
_hit = json.loads(search_regulation("gdpr breach notification", "GDPR"))
_ho = make_handoff("researcher", "writer",
                   [{"claim": _hit["passage"], "article_ids": _hit["article_ids"]}], "high")
print("handoff valid:", assert_handoff(_ho, _hit["article_ids"]))

handoff valid: True


### The team, by hand
A simple mediator passes structured messages between researcher and writer. The full trace is
printed with speaker labels, so debugging the team is reading a protocol.

In [4]:
def run_team_by_hand(task, verbose=True):
    """Two agents, no framework: researcher -> (validated handoff) -> writer."""
    trace = []
    # researcher acts
    r_out = json.loads(role_llm("researcher", "You research regulation. Hand off findings only.", {"task": task}))
    tool_ids = [aid for f in r_out["findings"] for aid in f["article_ids"]]
    trace.append(("researcher", f"findings: {r_out['findings']}"))
    # build + validate the handoff (Assert v3 at the team level)
    handoff = make_handoff("researcher", "writer", r_out["findings"], r_out["confidence"])
    assert_handoff(handoff, tool_ids)
    trace.append(("mediator", "handoff validated -> writer"))
    # writer acts on the findings only
    draft = role_llm("writer", "You write from findings only. Never invent facts.",
                     {"findings": handoff["findings"]})
    trace.append(("writer", f"draft: {draft}"))
    if verbose:
        for speaker, msg in trace:
            print(f"[{speaker:<10}] {msg}")
    return {"draft": draft, "trace": trace}

result = run_team_by_hand("Compare the breach-notification duty under GDPR.")
print("\nFINAL DRAFT:", result["draft"])
# trajectory assert: the writer's draft must rest on the researcher's findings
assert "Article 33" in result["draft"]
print("trajectory assert passed: draft is grounded in findings.")

[researcher] findings: [{'claim': 'Article 33: notify within 72 hours of awareness.', 'article_ids': [33, 34]}]
[mediator  ] handoff validated -> writer
[writer    ] draft: Under GDPR Article 33: notify within 72 hours of awareness. (see articles [33, 34]).

FINAL DRAFT: Under GDPR Article 33: notify within 72 hours of awareness. (see articles [33, 34]).
trajectory assert passed: draft is grounded in findings.


## Part B · LangGraph enters

### The same pair, as an explicit graph
The manual loop was a graph all along. We rebuild researcher + writer as LangGraph nodes with a
shared `state` and one conditional edge: if the research is insufficient, go back.

In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

class TeamState(TypedDict):
    task: str
    findings: list
    draft: str
    research_rounds: Annotated[int, operator.add]   # accumulates across visits

def researcher_node(state: TeamState) -> dict:
    out = json.loads(role_llm("researcher", "Research regulation; hand off findings.", state))
    return {"findings": out["findings"], "research_rounds": 1}

def writer_node(state: TeamState) -> dict:
    draft = role_llm("writer", "Write from findings only.", {"findings": state["findings"]})
    return {"draft": draft}

def research_sufficient(state: TeamState) -> str:
    """Conditional edge: enough findings to write, or go back for more?"""
    enough = bool(state.get("findings")) or state.get("research_rounds", 0) >= 2
    return "writer" if enough else "researcher"

print("nodes and conditional-edge router defined.")

nodes and conditional-edge router defined.


### Compile and run the graph
We wire the nodes, set the conditional edge, compile, and run. The state is printed at each node,
so the graph is observable end to end.

In [6]:
graph = StateGraph(TeamState)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.set_entry_point("researcher")
graph.add_conditional_edges("researcher", research_sufficient,
                            {"researcher": "researcher", "writer": "writer"})
graph.add_edge("writer", END)
app = graph.compile()

# run with a light tracer on the state
final = app.invoke({"task": "Compare GDPR breach notification.",
                    "findings": [], "draft": "", "research_rounds": 0})
print("research rounds:", final["research_rounds"])
print("draft:", final["draft"])

# same behaviour as the by-hand team, now as an explicit graph
assert "Article 33" in final["draft"]
assert final["research_rounds"] >= 1
print("\nsame result as Demo A, now an explicit LangGraph.")

research rounds: 1
draft: Under GDPR Article 33: notify within 72 hours of awareness. (see articles [33, 34]).

same result as Demo A, now an explicit LangGraph.


### The conditional edge, fired once
We force the "insufficient research" branch a single time to watch the graph loop back, exactly the
manual loop from S42 drawn as a cycle.

In [7]:
# a researcher that returns empty on its first visit, then finds on the second
_visits = {"n": 0}
def flaky_researcher(state: TeamState) -> dict:
    _visits["n"] += 1
    if _visits["n"] == 1:
        return {"findings": [], "research_rounds": 1}      # insufficient: edge sends it back
    out = json.loads(role_llm("researcher", "retry", state))
    return {"findings": out["findings"], "research_rounds": 1}

g2 = StateGraph(TeamState)
g2.add_node("researcher", flaky_researcher)
g2.add_node("writer", writer_node)
g2.set_entry_point("researcher")
g2.add_conditional_edges("researcher", research_sufficient,
                         {"researcher": "researcher", "writer": "writer"})
g2.add_edge("writer", END)
app2 = g2.compile()

final2 = app2.invoke({"task": "Compare GDPR breach notification.",
                      "findings": [], "draft": "", "research_rounds": 0})
print("research rounds (looped back once):", final2["research_rounds"])
assert final2["research_rounds"] == 2      # went back exactly once
print("conditional edge fired: the writer sent the researcher back for more.")

research rounds (looped back once): 2
conditional edge fired: the writer sent the researcher back for more.


### The cost of topology (delivered with transparency)
A team is not free. We show the naive view (which misleadingly looks cheaper), then the three
overhead sources that make a real team cost more. These are the numbers behind the S46 prediction.

In [8]:
def cost(cin, cout): return cin * PRICE_IN + cout * PRICE_OUT

# baseline: a single agent doing the whole comparison report
single = cost(2505, 290)

# NAIVE view (the trap): split work, ignore overhead -> looks cheaper
naive_team = cost(410+835, 45+260) + cost(320+90+260, 240)

# REAL team: three overhead sources counted explicitly
FULL_PERSONA, HANDOFF = 520, 260
real_researcher = cost((FULL_PERSONA+90)+(FULL_PERSONA+90+HANDOFF), 45+260)
real_writer     = cost(FULL_PERSONA+90+HANDOFF, 240)
orchestrator    = cost((FULL_PERSONA+120)+(FULL_PERSONA+120+HANDOFF), 30+30)
real_team = real_researcher + real_writer + orchestrator

# crew of 3: add a judge edge + one more routing hop
crew = real_team + cost(FULL_PERSONA+240+HANDOFF, 90) + cost(FULL_PERSONA+120+HANDOFF, 30)

for name, c in [("single agent", single), ("naive team (trap)", naive_team),
                ("real team", real_team), ("crew of 3", crew)]:
    print(f"{name:<18} ${c*1000:>5.2f} / 1k reports   {c/single:>4.1f}x")

print("\nthree overhead sources: full persona per role, handoff re-carries context, orchestrator routing")
# assert on the SHAPE: naive misleads, real team costs more, crew more still
assert naive_team < single < real_team < crew
print("topology costs money: every edge is an API call carrying context.")

single agent       $ 1.71 / 1k reports    1.0x
naive team (trap)  $ 1.56 / 1k reports    0.9x
real team          $ 2.77 / 1k reports    1.6x
crew of 3          $ 4.00 / 1k reports    2.3x

three overhead sources: full persona per role, handoff re-carries context, orchestrator routing
topology costs money: every edge is an API call carrying context.


## Transfer to your data (home bonus)
Step 2 of the assignment: add a live `arxiv_search` tool to your personal agent. Then split it into
a researcher (arxiv_search, read-only) and a writer, with a validated handoff, exactly as here.

In [9]:
# TODO (home): add a live arxiv_search tool, then split your agent into two roles:
#   researcher: has arxiv_search (read-only), hands off structured findings
#   writer:     has no tools, drafts from findings only
# Validate the handoff with an assert, like assert_handoff above.
print("Your team, your roles. See the assignment brief for Step 2.")

Your team, your roles. See the assignment brief for Step 2.
